In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

In [2]:
dfRead = pd.read_csv(r"Analyse/data.csv")
#print (len(dfRead))
#print (dfRead.columns.tolist())
spaltenList = ['datum', 'stunde', 'temperatur', 'luftfeuchtigkeit',  'windgeschwindigkeit', 'luftdruck', 'niederschlagshoehe_mm', 'gesamtbewoelkung', 'no2','o3','pm10' ]
dfO = dfRead[spaltenList].copy()
#print (df.columns.tolist())
#print(dfO.index)
#print(dfO.dtypes)
#print (dfO.describe ())
print ("END")

END


In [14]:

df = dfO.copy()
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. DATEN VORBEREITEN & SORTIEREN
df = df[df['datum'] >= '2008-10-01']
df["timestamp"] = pd.to_datetime(df["datum"]) + pd.to_timedelta(df["stunde"], unit="h")
df.set_index("timestamp", inplace=True)
df = df.sort_index()

# 2. FEATURING: Zeitbasierte Merkmale extrahieren
df["monat"] = df.index.month
df["wochentag"] = df.index.dayofweek
df["ist_wochenende"] = df["wochentag"].isin([5, 6]).astype(int)

# Basis-Wetterfeatures
base_features = [
    "stunde", "monat", "ist_wochenende", "temperatur", "luftfeuchtigkeit", 
    "windgeschwindigkeit", "luftdruck", "niederschlagshoehe_mm", "gesamtbewoelkung"
]

# Schadstoff-Liste definieren
stoffList = ['no2', 'o3', 'pm10']

# Lags für ALLE drei Schadstoffe generieren
lag_cols = []
for stoff in stoffList:
    df[f'{stoff}_lag_1h'] = df[stoff].shift(1)
    df[f'{stoff}_lag_2h'] = df[stoff].shift(2)
    df[f'{stoff}_lag_24h'] = df[stoff].shift(24)
    df[f'{stoff}_roll_mean_6h'] = df[stoff].shift(1).rolling(window=6).mean()
    
    lag_cols.extend([f'{stoff}_lag_1h', f'{stoff}_lag_2h', f'{stoff}_lag_24h', f'{stoff}_roll_mean_6h'])

# Alle Features kombinieren (Wetter + Lags)
feature_cols = base_features + lag_cols

# Ergebnisse speichern
ergebnisse = {}

# Schleife für das Training und die Evaluation aller Schadstoffe
for stoff in stoffList:
    print(f"--- Vorbereitung und Training für Schadstoff: {stoff.upper()} ---")
    
    # JETZT REINIGEN WIR SPEZIFISCH FÜR DEN AKTUELLEN STOFF
    # Wir brauchen: Gültige Features (lag_cols) UND eine gültige Zielvariable (stoff)
    benoetigte_spalten = [stoff] + lag_cols
    
    df_clean = df.dropna(subset=benoetigte_spalten)
    df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna(subset=benoetigte_spalten)
    
    # Features (X) und Zielvariable (y) für DIESEN Stoff extrahieren
    X_stoff = df_clean[feature_cols]
    y_stoff = df_clean[stoff]
    
    # Chronologischer Split (80% Train, 20% Test) auf den bereinigten Daten
    split_idx = int(len(df_clean) * 0.8)
    
    X_train, X_test = X_stoff.iloc[:split_idx], X_stoff.iloc[split_idx:]
    y_train, y_test = y_stoff.iloc[:split_idx], y_stoff.iloc[split_idx:]
    
    # Modell initialisieren und trainieren
    model = XGBRegressor(
        n_estimators=300, 
        learning_rate=0.05, 
        max_depth=5, 
        subsample=0.8, 
        colsample_bytree=0.8, 
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # Vorhersage und Metriken
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Ergebnisse speichern
    ergebnisse[stoff] = {"MAE": mae, "R²": r2}
    print(f"{stoff.upper()} - MAE: {mae:.2f} | R²: {r2:.2f}\n")

# Finale Übersicht ausgeben
print("=== ZUSAMMENFASSUNG ALLER MODELLE ===")
for stoff, metrics in ergebnisse.items():
    print(f"{stoff.upper()}: R² = {metrics['R²']:.2f}, MAE = {metrics['MAE']:.2f}")
print("END")


--- Vorbereitung und Training für Schadstoff: NO2 ---
NO2 - MAE: 4.90 | R²: 0.73

--- Vorbereitung und Training für Schadstoff: O3 ---
O3 - MAE: 4.94 | R²: 0.94

--- Vorbereitung und Training für Schadstoff: PM10 ---
PM10 - MAE: 4.99 | R²: 0.53

=== ZUSAMMENFASSUNG ALLER MODELLE ===
NO2: R² = 0.73, MAE = 4.90
O3: R² = 0.94, MAE = 4.94
PM10: R² = 0.53, MAE = 4.99
END


In [4]:
wetter_szenarien = pd.DataFrame(
    [
        {
            "stunde": 8,
            "monat": 1,
            "ist_wochenende": 0,
            "temperatur": 5.0,
            "luftfeuchtigkeit":np.nan,
            "windgeschwindigkeit": 6.5,  # Starker Wind (Schadstoffe werden verweht)
            "luftdruck": 1005.0,  # Tiefdruck
            "niederschlagshoehe_mm": 2.0,  # Regen wäscht Luft aus
            "gesamtbewoelkung": 8,
        }
    ]
)

# 2. Vorhersage durchführen
# Das Modell berechnet nun für jedes Szenario den NO2-Wert
vorhergesagte_werte = model.predict(wetter_szenarien)

# 3. Ergebnisse übersichtlich anzeigen
wetter_szenarien["vorhergesagtes_no2"] = vorhergesagte_werte
print(wetter_szenarien[["windgeschwindigkeit", "niederschlagshoehe_mm", "vorhergesagtes_no2"]])



   windgeschwindigkeit  niederschlagshoehe_mm  vorhergesagtes_no2
0                  6.5                    2.0           31.162392
